# 06 Recommendation System

This notebook implements the recommendation challenge using collaborative filtering. The dataset is suitable for this task because it contains explicit user-item-rating triplets: `username` identifies the user, `anime_id` identifies the item, and `score` provides the observed preference signal.

Unlike the supervised notebooks, the recommendation task uses the interaction matrix itself. The goal is not only to predict a global score from metadata, but to estimate how a specific user might rate anime that they have not yet rated. This makes recommendation especially appropriate for MyAnimeList, where individual taste is central.


In [ ]:
import os
import sys
import time

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append(os.path.abspath('..'))

RANDOM_STATE = 42
SAMPLE_SIZE = 50000

%matplotlib inline
sns.set_theme(style='whitegrid')


In [ ]:
from sklearn.model_selection import train_test_split

from src.evaluate import compute_metrics
from src.recommender import SimpleSVD, get_top_n_recommendations, simple_cross_validate


## Data Preparation

The recommendation dataset is reduced to the three fields required for collaborative filtering: user, item, and rating. Missing values are removed because matrix factorisation requires observed ratings for training. Duplicate user-item pairs are resolved by keeping the last record, ensuring that each interaction contributes one target value.

The sample size is limited for computational feasibility. Collaborative filtering can be expensive when there are many users and anime items, especially if the user-item matrix is sparse. The resulting table represents a sparse ratings matrix: most users rate only a small subset of all available anime, which is typical for recommendation systems.


In [ ]:
df = pd.read_csv('../datasets/ratings.csv')
df = df.sample(n=min(20000, len(df)), random_state=RANDOM_STATE)
df_recs = (
    df[['username', 'anime_id', 'score']]
    .dropna()
    .drop_duplicates(subset=['username', 'anime_id'], keep='last')
    .copy()
)
df_recs['score'] = df_recs['score'].astype(float)
display(df_recs.head())
print('Recommendation rows:', len(df_recs))


## Cross Validation with SimpleSVD

The recommender uses a matrix factorisation approach inspired by Funk SVD. The mathematical idea is to represent each user and each anime as vectors in a shared latent factor space. A predicted rating is obtained by combining the user vector and item vector, usually through their dot product plus bias terms or a global mean. Latent factors can capture hidden preference dimensions, such as genre affinity, tolerance for long series, preference for popular titles, or taste for specific formats.

Training is performed with gradient descent. For each observed rating, the model compares the predicted score with the true score, computes an error, and updates the user and item factors to reduce that error. Regularisation discourages excessively large factor values and helps prevent overfitting, which is important because the rating matrix is sparse.

The cross-validation RMSE and MAE estimate how well the recommender predicts held-out ratings. On a 1 to 10 rating scale, an MAE near one point would generally be interpretable as useful, while substantially larger errors would indicate weak personalisation. RMSE is more sensitive to large mistakes, so it is useful for detecting whether the model sometimes produces poor predictions even if the average absolute error is acceptable.


In [ ]:
cv_results = simple_cross_validate(df_recs, n_factors=20, cv=5)
print('CV RMSE mean:', cv_results['test_rmse'].mean())
print('CV MAE mean:', cv_results['test_mae'].mean())


## Train/Test Evaluation and MLflow Tracking

The train-test evaluation fits a SimpleSVD model with `n_factors=20`, `n_epochs=10`, learning rate `0.005`, and regularisation `0.02`. The number of factors controls the dimensionality of the latent preference space. More factors can capture more nuanced taste patterns but increase overfitting risk and training time. The number of epochs controls how many passes gradient descent makes over the training data. The learning rate determines update size, and regularisation controls model complexity.

These parameter values are moderate choices for an educational implementation: they are large enough to learn meaningful latent structure while keeping computation manageable. MLflow tracking records the model settings, fit time, RMSE, MAE, and artifacts, which supports reproducibility and comparison with future recommender variants.

The cold-start problem is an important limitation. If a user or anime was not present during training, the model does not have learned latent factors for it. The implementation handles this by falling back toward the global mean, which provides a safe default but not personalised recommendations. In a production system, cold-start cases would benefit from popularity priors, content metadata, or onboarding questionnaires.


In [ ]:
train_df, test_df = train_test_split(df_recs, test_size=0.2, random_state=RANDOM_STATE)
svd = SimpleSVD(n_factors=20, n_epochs=10, lr=0.005, reg=0.02, random_state=RANDOM_STATE)

while mlflow.active_run():
    mlflow.end_run()
with mlflow.start_run(nested=True, run_name='06_recommendation_svd'):
    start = time.time()
    svd.fit(train_df)
    fit_time = time.time() - start
    predictions = [svd.predict(row.username, row.anime_id).est for row in test_df.itertuples(index=False)]
    metrics = compute_metrics(test_df['score'], predictions)
    mlflow.log_param('algorithm', 'SimpleSVD')
    mlflow.log_param('n_factors', svd.n_factors)
    mlflow.log_param('n_epochs', svd.n_epochs)
    mlflow.log_metric('fit_time', fit_time)
    for key, value in metrics.items():
        mlflow.log_metric(key, value)

display(pd.DataFrame([{'fit_time': fit_time, **metrics}]))


## Top-N Recommendations

The Top-N section generates recommendations for an example user by predicting scores for candidate anime and ranking them by estimated preference. The resulting table should be interpreted as a personalised ranking: higher predicted scores indicate anime that the model expects this user to like more, based on patterns learned from similar rating behaviour.

These recommendations are collaborative rather than content-based. The model does not need to know genres or studios explicitly; it learns from the rating matrix. This is powerful when many users have overlapping histories, but it also means that recommendations may be weak for rare anime, new anime, or users with very few ratings.


In [ ]:
sample_user = train_df['username'].iloc[0]
recommendations = get_top_n_recommendations(svd, sample_user, train_df, n=10)
df_recommendations = pd.DataFrame(recommendations, columns=['anime_id', 'predicted_score'])
df_recommendations.insert(0, 'username', sample_user)
display(df_recommendations)


## Critical Interpretation

Matrix factorisation is appropriate for MyAnimeList because user taste is highly individual and because explicit ratings provide direct preference signals. Compared with content-based filtering, collaborative filtering can discover latent similarities that are not obvious from metadata alone. For example, two anime may be recommended together because the same users enjoy them, even if their genres or formats differ.

However, collaborative filtering also has limitations. It depends on sufficient overlap between users and items, so sparse interactions reduce reliability. It can reinforce popularity bias because well-rated popular anime have more training data. It also struggles with cold-start users and new anime, where no historical ratings exist. Content-based filtering could address some of these limitations by using genres, studios, synopsis text, or release metadata, but it may miss community preference patterns captured by collaborative filtering.

The recommender should therefore be understood as a feasible and justified challenge extension, not as a complete production recommendation system. A stronger final system could combine collaborative filtering with content features in a hybrid model.
